In [ ]:
import pandas as pd
import numpy as np

Obsługa **indeksowania hierarchicznego** jest ważnym elementem biblioteki pandas umożliwiającym
przypisanie do jednej osi wielu poziomów indeksowania (przypisanie dwóch lub więcej indeksów).

In [ ]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])

In [ ]:
data

a  1    0.128734
   2    0.634292
   3    0.269568
b  1    0.874453
   3    0.641675
c  1    0.749544
   2    0.782996
d  2    0.795548
   3    0.410639
dtype: float64

Wyświetloną czytelnie serią, której indeksem jest obiekt MultiIndex.

In [ ]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

Obiekty o indeksie hierarchicznym obsługują tzw. **indeksowanie częściowe**. Indeksowanie to umożliwia zwięzły wybór podzbioru danych

In [ ]:
data["b"]

In [ ]:
data["b":"c"]

b  1    0.874453
   3    0.641675
c  1    0.749544
   2    0.782996
dtype: float64

In [ ]:
data.loc[["b", "d"]]

## Przykład na danych

In [ ]:
# Dane sprzedażowe w dwóch miastach, w dwóch latach
df = pd.DataFrame({
    "miasto":   ["Łódź", "Łódź", "Kraków", "Kraków"],
    "rok":      [2023,   2024,   2023,     2024],
    "sprzedaz": [120,    145,    200,      230]
})
df

In [ ]:
df.index

RangeIndex(start=0, stop=4, step=1)

### Zwykły filtrowanie

Za każdym razem: filtr po mieście, filtr po roku, wybór kolumny. Trzy operacje dla jednej wartości.

In [ ]:
# Sprzedaż w Łodzi w 2024 - działa, ale rozwlekle:
df[(df.miasto == "Łódź") & (df.rok == 2024)]["sprzedaz"]

1    145
Name: sprzedaz, dtype: int64

In [ ]:
# Sprzedaż w Krakowie w 2023 - znowu to samo:
df[(df.miasto == "Kraków") & (df.rok == 2023)]["sprzedaz"]

2    200
Name: sprzedaz, dtype: int64

### MultiIndex - indeks jako "ścieżka"

In [ ]:
# Ten sam zbiór, ale z hierarchicznym indeksem:
sprzedaz = pd.Series(
    [120, 145, 200, 230],
    index=[["Łódź", "Łódź", "Kraków", "Kraków"],
           [2023,   2024,   2023,     2024]]
)
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [ ]:
# Sam indeks to obiekt MultiIndex:
sprzedaz.index

MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           )

### Indeksowanie - podstawowe wzorce

In [ ]:
# Pojedyncza wartość - krotka (miasto, rok):
sprzedaz["Łódź", 2024]

np.int64(145)

In [ ]:
# Wszystko dla Łodzi (wybór zewnętrznego poziomu):
sprzedaz["Łódź"]

2023    120
2024    145
dtype: int64

In [ ]:
# Wszystkie miasta, ale tylko rok 2023 (wewnętrzny poziom):
sprzedaz.loc[:, 2023]

Łódź      120
Kraków    200
dtype: int64

In [ ]:
# Zakres miast (uwaga: wymaga posortowanego indeksu):
sprzedaz.sort_index().loc["Kraków":"Łódź"]

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64

## Z Series do DataFrame i z powrotem

In [ ]:
print(sprzedaz)

Łódź    2023    120
        2024    145
Kraków  2023    200
        2024    230
dtype: int64


In [ ]:
# unstack - "wypchnięcie" wewnętrznego poziomu do kolumn:
tabela = sprzedaz.unstack()
print(tabela)
print(type(tabela))

        2023  2024
Kraków   200   230
Łódź     120   145
<class 'pandas.core.frame.DataFrame'>


In [ ]:
# stack - operacja odwrotna: kolumny wracają do indeksu:
tab_stack = tabela.stack()
print(tab_stack)
print(type(tab_stack))

Kraków  2023    200
        2024    230
Łódź    2023    120
        2024    145
dtype: int64
<class 'pandas.core.series.Series'>
MultiIndex([('Kraków', 2023),
            ('Kraków', 2024),
            (  'Łódź', 2023),
            (  'Łódź', 2024)],
           )


In [ ]:
print(df)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


In [ ]:
# Konwersja płaskiej ramki na hierarchiczną - set_index:
df_hier = df.set_index(["miasto", "rok"])
print(df_hier)
print(df_hier.index)

             sprzedaz
miasto rok           
Łódź   2023       120
       2024       145
Kraków 2023       200
       2024       230
MultiIndex([(  'Łódź', 2023),
            (  'Łódź', 2024),
            ('Kraków', 2023),
            ('Kraków', 2024)],
           names=['miasto', 'rok'])


In [ ]:
# I z powrotem - reset_index:
df2 = df_hier.reset_index()
print(df2)

   miasto   rok  sprzedaz
0    Łódź  2023       120
1    Łódź  2024       145
2  Kraków  2023       200
3  Kraków  2024       230


### Tabele przestawne (ang. _Pivot Table_)

In [ ]:
# Przekształcamy płaską ramkę w tabelę przestawną
df2_my_pivot = df.set_index(["miasto", "rok"])["sprzedaz"].unstack()
print(df2_pivot)

rok     2023  2024
miasto            
Kraków   200   230
Łódź     120   145


In [ ]:
df_pivot = pd.pivot_table(df,
                         index="miasto",
                         columns="rok",
                         values="sprzedaz")
print(df_pivot)

rok      2023   2024
miasto              
Kraków  200.0  230.0
Łódź    120.0  145.0


In [ ]:
df_kwartaly = pd.DataFrame({
    "miasto":   ["Łódź"]*4 + ["Kraków"]*4,
    "rok":      [2023]*4 + [2023]*4,
    "kwartal":  ["Q1","Q2","Q3","Q4"] * 2,
    "sprzedaz": [30, 25, 35, 30, 50, 45, 55, 50]
})

df_kwartaly_sum = pd.pivot_table(df_kwartaly,
                                index="miasto",
                                columns="rok",
                                values="sprzedaz",
                                aggfunc="sum")
print(df_kwartaly_sum)

rok     2023
miasto      
Kraków   200
Łódź     120


### Agregacje po poziomie

In [ ]:
sprzedaz.index.names = ["miasto", "rok"]
sprzedaz.groupby(level="miasto").sum()

miasto
Kraków    430
Łódź      265
dtype: int64

In [ ]:
sprzedaz.groupby(level="rok").mean()

rok
2023    160.0
2024    187.5
dtype: float64

### Agregacja w DataFrame z MultiIndex

In [ ]:
frame = pd.DataFrame({
    "sprzedaz": [120, 145, 200, 230],
    "koszty":   [80,  90,  130, 140]
}, index=[["Łódź","Łódź","Kraków","Kraków"],
          [2023,  2024,  2023,    2024]])
frame.index.names = ["miasto", "rok"]
frame.groupby(level="miasto").sum()

,sprzedaz,koszty
miasto,,
Kraków,430,270
Łódź,265,170


In [ ]:
frame.groupby(level="miasto").agg(["sum", "mean"])

sprzedaz        koszty       
            sum   mean    sum   mean
miasto                              
Kraków      430  215.0    270  135.0
Łódź        265  132.5    170   85.0

In [ ]:
frame["sprzedaz"].groupby(level="miasto").agg(
    total="sum", srednia="mean", rozstep=lambda x: x.max()-x.min()
    )

,total,srednia,rozstep
miasto,,,
Kraków,430,215.0,30
Łódź,265,132.5,25


# *Zadania*

Sieć "ModaŁódź" ma sklepy w trzech miastach. Każdy sklep sprzedaje trzy kategorie produktów. Dane obejmują 4 kwartały 2023 i 2024 roku.

## **Zadanie 1 — Budowanie MultiIndex**

a) Utwórz ramkę `df_hi` z hierarchicznym indeksem (`miasto`, `kategoria`, `rok`, `kwartal`). Posortuj indeks.

b) Wyświetl liczbę poziomów indeksu i ich nazwy.

c) Ile unikalnych kombinacji indeksu istnieje? Użyj `.index` do odpowiedzi.

## **Zadanie 2 — Selekcje na MultiIndex**

a) Wybierz wszystkie dane dla Łodzi.
    
b) Wybierz dane dla Łodzi, kategorii Damska.
    
c) Wybierz dane dla wszystkich miast, ale tylko rok 2024. (Wskazówka: użyj metodę `xs()`)

d) Wybierz Q4 z obu lat, ale tylko dla Krakowa i kategorii Męska.

## **Zadanie 3 — pivot_table: roczne podsumowanie**

a) Utwórz tabelę przestawną `tab_miasta`, która pokaże **sumę przychodów** w wierszach per miasto, w kolumnach per rok.

b) Dodaj do tabeli kolumnę `zmiana_proc` — procentową zmianę przychodu między 2023 a 2024. Które miasto rosło najszybciej?

c) Utwórz drugą tabelę przestawną, w której wiersze to (miasto, kategoria), kolumny to rok, wartości to *średni przychód kwartalny*. Która kombinacja (miasto, kategoria) ma najwyższy średni przychód w 2024?

## **Zadanie 4 — Agregacja po poziomach**
a) Na `df_hi` oblicz sumę przychodów per miasto (agreguj po poziomie `"miasto"`).

b) Oblicz **średni przychód kwartalny per kategoria** (agreguj po poziomie `"kategoria"`).

c) Użyj `.agg(["sum", "mean", "max"])` na kolumnie `przychod_tys` pogrupowanej po `(miasto, rok)`. Który wiersz ma najwyższą wartość `max`?

## **Zadanie 5 — Marża i ranking**
a) W `df_hi` dodaj kolumnę `marza_tys = przychod_tys − koszt_tys`.

b) Utwórz tabelę przestawną: *wiersze = miasto*, *kolumny = kategoria*, *wartości = suma marży*. Które miasto jest najbardziej zyskowne? Która kategoria generuje najwyższą marżę?